In [1]:
!pip install roboflow
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 39.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 86.3 MB/s eta 0:00:00:00:01
  Attempting uninstall: idna
    Found existing installation: idna 3.10
    Uninstalling idna-3.10:
      Successfully uninstalled idna-3.10
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.11.0.86
    Uninstalling opencv-python-headless-4.11.0.86:
      Successfully uninstalled opencv-python-headless-4.11.0.86
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.8.0 requires google-cloud-bigquery-sto

In [2]:
from roboflow import Roboflow
from ultralytics import YOLO
import cv2
import os
import torch
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
api_key = "UYTUzKsMZToy79NfRukz"#os.environ["API_KEY"]

In [4]:
class KeyPointDetectionModel():
    def __init__(self, api_key, workspace, project_id):
        self.api_key = api_key
        self.workspace = workspace
        self.project_id = project_id
    
    def start(self):
        rf = Roboflow(self.api_key)
        project = rf.workspace(self.workspace).project(self.project_id)
        return project

In [5]:
session = KeyPointDetectionModel(api_key, "cvbnqq", "dataset-ridimensionato-b9tds")
project = session.start()
print(project)

loading Roboflow workspace...
loading Roboflow project...
{
  "name": "dataset ridimensionato",
  "type": "keypoint-detection",
  "workspace": "cvbnqq"
}


In [6]:
model_path = "/kaggle/input/physical-excersies-model/pytorch/default/1/best-n-09.pt"
model = YOLO(model_path)

In [7]:
model.task

'pose'

In [8]:
import cv2
import numpy as np
from ultralytics import YOLO
import math

def count_squats(model_path, video_path, output_path="squats_cnt.mp4"):
    model = YOLO(model_path)
    cap = cv2.VideoCapture(video_path, cv2.CAP_FFMPEG)

    if not cap.isOpened():
        print("video open error")
    

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    squat_count = 0
    is_down_position = False
    squat_threshold_up = 160  
    squat_threshold_down = 120 
    confidence_threshold = 0.3
    
    LEFT_GROIN = 15   
    LEFT_KNEE = 17    
    LEFT_ANKLE = 18  
    
    keypoint_names = {
        15: "left_gorin",
        17: "left_knee",
        18: "left_anckle"
    }
    
    frame_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_count += 1
        results = model(frame, verbose = False, conf=confidence_threshold)
        
        if len(results) > 0 and results[0].keypoints is not None:
            result = results[0]

            for keypoints in result.keypoints:
                if keypoints.conf is None:
                    continue
                
                keypoints_data = {}
                valid_points = 0
                
                for idx in [LEFT_GROIN, LEFT_KNEE, LEFT_ANKLE]:
                    if idx < len(keypoints.xy[0]) and idx < len(keypoints.conf[0]):
                        conf = keypoints.conf[0][idx].item()
                        if conf > confidence_threshold:
                            x = int(keypoints.xy[0][idx][0].item())
                            y = int(keypoints.xy[0][idx][1].item())
                            keypoints_data[idx] = (x, y, conf)
                            valid_points += 1
                
                if valid_points == 3:
                    point_a = keypoints_data[LEFT_GROIN][:2]  
                    point_b = keypoints_data[LEFT_KNEE][:2]   
                    point_c = keypoints_data[LEFT_ANKLE][:2]  
                    
                    ba = np.array(point_a) - np.array(point_b)
                    bc = np.array(point_c) - np.array(point_b)
                    
                    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
                    cosine_angle = np.clip(cosine_angle, -1.0, 1.0)
                    angle = math.degrees(math.acos(cosine_angle))
                    
                    for idx, (x, y, conf) in keypoints_data.items():
                        cv2.circle(frame, (x, y), 6, (0, 255, 0), -1)
                    
                    if angle < squat_threshold_down and not is_down_position:
                        is_down_position = True
                        cv2.putText(frame, "DOWN", (50, 100),
                                   cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
                    
                    elif angle > squat_threshold_up and is_down_position:
                        squat_count += 1
                        is_down_position = False
                        cv2.putText(frame, "UP", (50, 150),
                                   cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 3)
                
                skeleton = [
                    [(0, 1), (0, 2), (1, 3), (1, 2)],  # голова
                    [(4,5), (5,6), (6,7), (7,8)],       # левая рука
                    [(9,10), (10,11), (11,12), (12,13)], # правая рука
                    [(3,4), (3,9), (3,14), (14,15), (14,16)],  # тело к тазу
                    [(15,17), (17,18), (18,19), (18,20)],  # левая нога
                    [(16,21), (21,22), (22,23), (22,24)]   # правая нога
                ]
                
                colors = [(0,0,255), (0,255,0), (255,0,0), (255,255,0), 
                         (255,0,255), (0,255,255)]
                
                for color_idx, bone_group in enumerate(skeleton):
                    for a, b in bone_group:
                        if (a < len(keypoints.xy[0]) and b < len(keypoints.xy[0]) and
                            a < len(keypoints.conf[0]) and b < len(keypoints.conf[0]) and
                            keypoints.conf[0][a].item() > confidence_threshold and
                            keypoints.conf[0][b].item() > confidence_threshold):
                            
                            pt1 = (int(keypoints.xy[0][a][0].item()), 
                                  int(keypoints.xy[0][a][1].item()))
                            pt2 = (int(keypoints.xy[0][b][0].item()), 
                                  int(keypoints.xy[0][b][1].item()))
                            
                            cv2.line(frame, pt1, pt2, colors[color_idx % len(colors)], 2)
        
        cv2.putText(frame, f"Squats: {squat_count}", (50, 50),
                   cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 255), 3)
        
        out.write(frame)
    
    cap.release()
    out.release()
    
    print(f"total squats: {squat_count}")
    
    return squat_count

In [9]:
count_squats(model_path, "/kaggle/input/squats/squats.mov", output_path="squats_cnt.mp4")

total squats: 2


2